# 03 — Exploratory Data Analysis
**Spacecraft Telemetry Anomaly Detection | Stage 1**

---
**Goal:** Understand what *normal* looks like — the baseline that anomalies will deviate from.

> EDA is not just visualisation — it builds the intuition needed to choose the right anomaly detection model and interpret its outputs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
                     'axes.titlesize':11, 'font.size':10})
os.makedirs('plots_v2', exist_ok=True)

PALETTE = ['#1f77b4','#2ca02c','#d62728','#9467bd','#8c564b',
           '#e377c2','#7f7f7f','#bcbd22','#17becf','#ff7f0e']

telemetry_raw   = pd.read_csv('telemetry_train.csv')
telecommand_raw = pd.read_csv('telecommand_train.csv')
telemetry_raw['timestamp']   = pd.to_datetime(telemetry_raw['timestamp'])
telecommand_raw['timestamp'] = pd.to_datetime(telecommand_raw['timestamp'])

params = sorted(telemetry_raw['parameter'].unique())
print(f'Loaded | Telemetry: {telemetry_raw.shape} | {len(params)} parameters | {telecommand_raw["command"].nunique()} commands')

### 3.1 Parameter & Command Frequency
How many times is each parameter/command recorded in the dataset?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

telemetry_raw['parameter'].value_counts().sort_values().plot(
    kind='barh', ax=axes[0], color='#1f77b4', edgecolor='white')
axes[0].set_title('Telemetry — Parameter Frequency', fontweight='bold')
axes[0].set_xlabel('Count')

telecommand_raw['command'].value_counts().sort_values().plot(
    kind='barh', ax=axes[1], color='#2ca02c', edgecolor='white')
axes[1].set_title('Telecommand — Command Frequency', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('plots_v2/03a_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

# RESULT: All 50 parameters have equal frequency (~200 each) — uniform round-robin polling
# Each of the 161 commands appears once — this is nominal command history, not repeated

### 3.2 Value Distributions — All 50 Parameters
Each histogram shows the spread of readings for one parameter. Red = mean, orange = median.

In [ ]:
ncols = 5
nrows = (len(params) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(22, nrows * 3.5))
fig.suptitle('Value Distributions — All 50 Parameters  (red=mean, orange=median)',
             fontsize=13, fontweight='bold', y=1.01)

for idx, param in enumerate(params):
    ax   = axes.flatten()[idx]
    data = telemetry_raw.loc[telemetry_raw['parameter'] == param, 'value']
    ax.hist(data, bins=25, color=PALETTE[idx % len(PALETTE)], alpha=0.8, edgecolor='white')
    ax.axvline(data.mean(),   color='red',    lw=1.3, ls='--')
    ax.axvline(data.median(), color='orange', lw=1.3, ls=':')
    ax.set_title(param, fontsize=7, fontweight='bold')
    ax.tick_params(labelsize=6)

for j in range(idx + 1, len(axes.flatten())):
    axes.flatten()[j].set_visible(False)

plt.tight_layout()
plt.savefig('plots_v2/03b_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# KEY RESULTS:
# - Most parameters show approximately normal (bell-shaped) distributions
#   --> Validates Gaussian-assumption models like Isolation Forest
# - Red (mean) ≈ orange (median) for most --> low skew, symmetric distributions
# - GYRO_X/Y/Z are very tightly centred near 0 --> spacecraft is well-stabilised
# - RF_SIGNAL_STRENGTH has wider spread --> expected from orbital geometry variation
# - REACTION_WHEEL_SPD_* show symmetric distribution around near-zero

### 3.3 Boxplots — Grouped by Subsystem
Boxplots reveal spread, IQR, and mild outliers within each spacecraft subsystem.

In [ ]:
# Group parameters by spacecraft subsystem for cleaner comparison
SUBSYSTEMS = {
    'Power':      [p for p in params if any(k in p for k in ['BATT','SOLAR','BUS'])],
    'Thermal':    [p for p in params if 'TEMP' in p or 'RADIATOR' in p],
    'ADCS':       [p for p in params if any(k in p for k in
                   ['GYRO','MAG','REACTION','ATTITUDE','STAR','SUN_SENSOR'])],
    'Comms':      [p for p in params if any(k in p for k in
                   ['RF','TX','LINK','DATA_RATE','PACKET'])],
    'OBC':        [p for p in params if any(k in p for k in
                   ['CPU','MEMORY','FLASH','WATCHDOG'])],
    'Propulsion': [p for p in params if any(k in p for k in
                   ['TANK','THRUSTER_VALVE','THRUSTER_TEMP'])],
}

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle('Boxplots by Subsystem  (red line = median | dots = mild outliers)',
             fontsize=13, fontweight='bold')

for ax, (subsys, ps), col in zip(axes.flatten(), SUBSYSTEMS.items(), PALETTE):
    valid_ps  = [p for p in ps if (telemetry_raw['parameter'] == p).any()]
    data_list = [telemetry_raw.loc[telemetry_raw['parameter'] == p, 'value'].values
                 for p in valid_ps]
    bp = ax.boxplot(data_list, labels=valid_ps, patch_artist=True,
                    medianprops=dict(color='red', lw=1.5),
                    flierprops=dict(marker='.', ms=3, alpha=0.5))
    for patch in bp['boxes']:
        patch.set_facecolor(col); patch.set_alpha(0.5)
    ax.set_title(subsys, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.tight_layout()
plt.savefig('plots_v2/03c_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

# KEY RESULTS:
# - Power: BATT_VOLTAGE_1/2 very tight boxes (stable charging) | SOLAR_POWER wider
# - Thermal: RADIATOR_TEMP has the widest range (deep-space radiative cycling)
# - ADCS: GYRO boxes extremely narrow (excellent attitude stability)
# - Comms: RF_SIGNAL_STRENGTH shows wider IQR (orbital viewing angle variation)
# - Dot outliers visible in normal data -- these are mild, not anomalies yet
# - Boxes confirm: parameters span wildly different physical unit ranges --> scaling needed

### 3.4 Time-Series — 8 Representative Parameters
We plot raw readings over time with a rolling mean overlay to reveal trends.

In [ ]:
# One parameter selected from each major subsystem
TS_PARAMS = ['BATT_VOLTAGE_1','SOLAR_POWER_TOTAL','OBC_TEMP','GYRO_X',
             'RF_SIGNAL_STRENGTH','ATTITUDE_ROLL','TANK_PRESSURE','MEMORY_USAGE']

fig, axes = plt.subplots(4, 2, figsize=(18, 14))
fig.suptitle('Time-Series  (blue=raw | red dashed=rolling mean of 10 readings)',
             fontsize=13, fontweight='bold')

for ax, param, col in zip(axes.flatten(), TS_PARAMS, PALETTE):
    sub = telemetry_raw[telemetry_raw['parameter'] == param].sort_values('timestamp')
    ax.plot(sub['timestamp'], sub['value'],
            color=col, lw=0.8, alpha=0.75, label='Raw')
    ax.plot(sub['timestamp'],
            sub['value'].rolling(10, center=True).mean(),  # smoothed trend
            color='red', lw=1.5, ls='--', label='RM(10)')
    ax.set_title(param, fontweight='bold', fontsize=9)
    ax.set_ylabel('Value', fontsize=8)
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('plots_v2/03d_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

# KEY RESULTS:
# - BATT_VOLTAGE_1 & SOLAR_POWER: gentle sinusoidal wave -- orbital day/night charging cycle
# - OBC_TEMP: slow rising trend -- normal thermal build-up during operations
# - GYRO_X: zero-mean, tiny oscillations -- excellent attitude stability
# - RF_SIGNAL_STRENGTH: more volatile -- spacecraft geometry relative to ground station changes
# - TANK_PRESSURE: very stable with tiny drift -- normal propellant consumption
# - Rolling mean (red) separates underlying trend from sensor noise
# --> Large deviations from the rolling mean will become our primary anomaly signal

### 3.5 Per-Parameter Descriptive Statistics

In [ ]:
param_stats = (telemetry_raw.groupby('parameter')['value']
               .agg(['mean','std','min','max',
                     lambda x: x.quantile(0.25),
                     lambda x: x.quantile(0.75)])
               .rename(columns={'<lambda_0>':'Q1', '<lambda_1>':'Q3'})
               .round(4))
display(param_stats)

# RESULT: std column shows natural variability per parameter
# Low std (e.g., GYRO_X ~0.03) = very stable parameter -- easy to spot anomalies
# High std (e.g., RF_SIGNAL_STRENGTH ~10) = naturally noisy -- harder to detect subtle anomalies
# Q1/Q3 define the IQR -- values outside 1.5×IQR are boxplot outliers